# Tabac et cancer du poumon : d'où viennent les écarts entre départements ?

*Projet Python pour la data science — ENSAE 2A — Jean Palazuelo, Ilyan Radjabaly*

## Introduction

Le tabac est la première cause de cancer du poumon, c'est établi depuis longtemps. Ce qui nous intéresse ici, c'est autre chose : la fréquence de ce cancer varie beaucoup d'un département à l'autre, et on voulait savoir si ces écarts se retrouvent dans la géographie du tabagisme.

Deux difficultés guident tout le projet :
- un cancer du poumon apparaît en général 20 à 30 ans après l'exposition, donc il faut comparer le cancer d'aujourd'hui au tabac d'avant ;
- les départements n'ont pas la même pyramide des âges, et ce cancer touche surtout les plus de 60 ans. Sans correction, on comparerait surtout l'âge des habitants.

**Question :** à âge égal, les départements les plus touchés par le cancer du poumon sont-ils ceux des régions où l'on fumait le plus ?

## 1. Les données

Trois sources, toutes téléchargées par le code (fonctions dans `src/data/`) et mises en cache dans `data/raw/` :

| Données | Source | Niveau | Années |
|---|---|---|---|
| Patients traités pour un cancer du poumon, par classe d'âge de 5 ans | Assurance Maladie, data.ameli.fr (API) | département et région | 2015-2024 |
| Part de fumeurs quotidiens chez les adultes, avec intervalle de confiance | Santé publique France, Baromètre santé (API Odissé) | région | 2000, 2005, 2010, 2014, 2017, 2021 |
| Contours des départements | france-geojson (IGN) | département | — |

À noter dès maintenant :
- la donnée Ameli compte les personnes **prises en charge** pour un cancer du poumon une année donnée (une prévalence), pas les nouveaux cas ni les décès ;
- le tabagisme n'existe qu'au **niveau régional**, et pas pour la Corse (échantillon trop petit).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from src.data.ameli import telecharger_cancer_poumon
from src.data.tabac import telecharger_tabac
from src.data.geo import telecharger_departements
from src.cleaning import taux_cancer, tabac_par_region, construire_table
from src.viz import cartes_cote_a_cote, nuage, evolution, nuage_ic, barres

ameli = telecharger_cancer_poumon()
tabac = telecharger_tabac()
contours = telecharger_departements()

print(ameli.shape, tabac.shape, contours.shape)
ameli.head()

La table Ameli a une ligne par année, département et classe d'âge : `ntop` est le nombre de patients, `npop` la population couverte. Les lignes `dept = "999"` donnent le total de chaque région, et `cla_age_5 = "tsage"` le total tous âges.

`ntop` est parfois vide : quand l'effectif est trop petit, l'Assurance Maladie ne le publie pas (secret statistique). Regardons où ça arrive :

In [ ]:
metro = ameli[~ameli["dept"].str.startswith("97") & (ameli["dept"] != "999") & (ameli["cla_age_5"] != "tsage")]
metro.groupby("cla_age_5")["ntop"].apply(lambda s: s.isna().mean()).round(2)

Les cases masquées sont presque toutes chez les moins de 40 ans, où il n'y a quasiment pas de cancers du poumon : les compter comme 0 ne change presque rien. C'est moins vrai pour les très âgés (près de la moitié des cases masquées chez les 90-94 ans, presque toutes chez les 95 ans et plus) : on sous-estime donc un peu le taux des petits départements, mais ces classes pèsent très peu une fois pondérées par la population.

## 2. Nettoyage : un taux comparable entre départements

On ne peut pas comparer des nombres de patients (le Nord a 2,5 millions d'habitants, la Lozère 76 000). On calcule donc un taux pour 100 000 habitants, et surtout un **taux standardisé sur l'âge** :

1. dans chaque département, on calcule le taux de chaque classe d'âge ;
2. on fait la moyenne de ces taux en les pondérant par la part de chaque classe d'âge dans la France métropolitaine.

On obtient le taux qu'aurait le département s'il avait la même pyramide des âges que la métropole. On fait la moyenne sur 2022-2024 pour lisser le hasard dans les petits départements (fonction `taux_cancer`).

Chaque département reçoit ensuite le tabagisme de sa région (`construire_table`). On prend le tabac de **2000**, la mesure la plus ancienne disponible.

In [ ]:
ANNEES_CANCER = [2022, 2023, 2024]

cancer_dep = taux_cancer(ameli, ANNEES_CANCER)
table = construire_table(cancer_dep, tabac_par_region(tabac, 2000), contours)

print(table.shape)
print(table.drop(columns="geometry").isna().sum())
table[["taux_brut", "taux_std"]].describe().round(0)

96 départements métropolitains, aucun manquant pour le cancer. Les deux départements corses n'ont pas de donnée de tabac, ils seront exclus des comparaisons avec le tabac.

La standardisation resserre fortement les écarts : l'écart-type passe de 36 à 20 pour 100 000. Une bonne partie des différences brutes venait donc simplement de l'âge.

## 3. Description

### Ce que change la standardisation

In [ ]:
fig = cartes_cote_a_cote(table, [
    dict(colonne="taux_brut", titre="Taux brut (2022-2024)", legende="patients pour 100 000 hab."),
    dict(colonne="taux_std", titre="Taux standardisé sur l'âge", legende="patients pour 100 000 hab."),
])

In [ ]:
ecart = table.assign(ecart=table["taux_brut"] - table["taux_std"]).sort_values("ecart")
cols = ["nom_dept", "taux_brut", "taux_std"]
pd.concat([ecart[cols].head(4), ecart[cols].tail(4)]).round(0)

Les deux cartes ne racontent pas la même histoire. Sur la carte brute, le centre de la France (Creuse, Nièvre, Lot, Cantal) ressort fortement : ce sont des départements très âgés, et une fois l'âge neutralisé ils rentrent dans la moyenne (le Cantal passe de 213 à 148). À l'inverse, la Seine-Saint-Denis et le Val-d'Oise, très jeunes, ont un taux brut faible (environ 120) mais un taux standardisé dans la moyenne (environ 160).

C'est la carte standardisée qu'on utilise pour la suite.

In [ ]:
table.sort_values("taux_std", ascending=False)[["nom_dept", "nom_region", "taux_std"]].head(8).round(0)

Les départements les plus touchés à âge égal sont la Corse, le Nord-Est (Meurthe-et-Moselle, Meuse), le pourtour méditerranéen (Hérault, Gard), l'Ariège et le Finistère. Le Nord, souvent cité pour le cancer du poumon, est seulement au milieu du classement avec cette mesure. Les moins touchés sont les Alpes (Hautes-Alpes, Savoie) et les Pays de la Loire (Sarthe).

### Évolution dans le temps

In [ ]:
noms = tabac[["region", "nom_region"]].drop_duplicates()
par_annee = pd.concat(
    [taux_cancer(ameli, [annee], niveau="region").assign(annee=annee) for annee in range(2015, 2025)]
).merge(noms, on="region")

ax = evolution(
    par_annee, "annee", "taux_std", "nom_region",
    "Cancer du poumon : taux standardisé par région", "Année", "Patients pour 100 000 hab.",
    a_souligner=["Grand Est", "Occitanie", "Hauts-de-France", "Pays de la Loire"],
)

Le taux augmente partout, d'environ 35 % entre 2015 et 2024 au niveau national. Ce n'est pas forcément une hausse des cancers : comme on mesure des patients **pris en charge**, de meilleurs traitements, qui prolongent la survie, font mécaniquement monter le nombre de patients suivis. On voit aussi un palier en 2020-2021, sans doute lié au Covid (moins de diagnostics et de suivis).

L'ordre entre régions, lui, est assez stable : le Grand Est et l'Occitanie restent en haut, les Pays de la Loire en bas.

### La géographie du tabac

In [ ]:
tabac_ensemble = tabac[tabac["sexe"] == "Hommes et Femmes"]
tabac_ensemble.pivot(index="nom_region", columns="annee", values="tabac_quotidien").round(1)

In [ ]:
fig = cartes_cote_a_cote(table, [
    dict(colonne="taux_std", titre="Cancer du poumon, taux standardisé (2022-2024)", legende="pour 100 000 hab."),
    dict(colonne="tabac_quotidien_2000", titre="Fumeurs quotidiens en 2000 (par région)", legende="% des adultes", cmap="Oranges"),
])

En 2000, entre 26 % (Pays de la Loire) et 34 % (Provence-Alpes-Côte d'Azur) des adultes fumaient tous les jours. Les écarts sont modestes, et les intervalles de confiance font environ ±3,5 points : beaucoup de régions ne sont pas statistiquement différentes.

Sur les cartes, certaines correspondances sautent aux yeux (PACA et le Grand Est fument beaucoup et ont des taux élevés ; les Pays de la Loire fument peu et ont des taux faibles), d'autres pas du tout : les Hauts-de-France fumaient plutôt **moins** que la moyenne en 2000.

## 4. Analyse : le tabac explique-t-il les écarts ?

### Au niveau des départements

In [ ]:
ax = nuage(
    table, "tabac_quotidien_2000", "taux_std",
    "Tabac en 2000 et cancer du poumon en 2022-2024, par département",
    "Fumeurs quotidiens en 2000, dans la région (%)", "Taux standardisé pour 100 000 hab.",
)

Le lien est faible (r = 0,18). Mais ce graphique est trompeur : le tabac n'a que 12 valeurs différentes, une par région, recopiée sur tous ses départements. D'où les colonnes verticales de points. Les écarts **à l'intérieur** d'une région (de 140 à plus de 200 en Occitanie) ne peuvent pas venir du tabac dans nos données, puisqu'il est le même par construction. Et on n'a en réalité que 12 observations indépendantes du tabac, pas 94.

La comparaison honnête se fait donc au niveau où le tabac est mesuré : la région.


### Au niveau des régions

In [ ]:
cancer_reg = taux_cancer(ameli, ANNEES_CANCER, niveau="region")
regions = cancer_reg.merge(tabac_par_region(tabac, 2000), on="region")

ax = nuage_ic(
    regions, "tabac_quotidien_2000", "taux_std", "tabac_ic_bas_2000", "tabac_ic_haut_2000", "nom_region",
    "Tabac en 2000 et cancer du poumon en 2022-2024, par région",
    "Fumeurs quotidiens en 2000 (%), avec intervalle de confiance à 95 %", "Taux standardisé pour 100 000 hab.",
)

Par région, la corrélation monte à 0,55. Une partie vient de l'agrégation elle-même : en faisant la moyenne de plusieurs départements, on efface leurs différences internes et il reste surtout la tendance commune. Le résultat repose aussi sur peu de points : les Pays de la Loire (peu de tabac, peu de cancers) et PACA (beaucoup des deux) tirent la droite, et les barres d'incertitude se chevauchent presque toutes.

### Quelle année de tabac colle le mieux ?

Si le délai de 20 à 30 ans compte, le tabac de 2000 devrait mieux coller au cancer d'aujourd'hui que celui de 2021. On compare les six éditions de l'enquête :

In [ ]:
correlations = pd.Series({
    annee: regions_annee["taux_std"].corr(regions_annee[f"tabac_quotidien_{annee}"])
    for annee in [2000, 2005, 2010, 2014, 2017, 2021]
    for regions_annee in [cancer_reg.merge(tabac_par_region(tabac, annee), on="region")]
})

ax = barres(
    correlations, "Corrélation entre le tabagisme de chaque année et le cancer 2022-2024 (12 régions)",
    "Année de mesure du tabagisme", "Corrélation r",
)

C'est l'inverse de ce qu'on attendait : le tabac de 2000 est celui qui colle le **moins** (0,55), les années suivantes sont toutes autour de 0,8. Plusieurs explications possibles, qu'on ne peut pas départager avec ces données :
- l'enquête de 2000 est moins précise (intervalles de confiance plus larges : 7 points en moyenne contre 5,7 en 2021) ;
- si une région fumait plus en 2000, elle a tendance à fumer plus encore aujourd'hui : le tabac récent sert alors d'indicateur de ce qu'était le tabac passé, avec moins de bruit ;
- 2000 n'est de toute façon pas assez ancien : les cancers de 2022-2024 viennent surtout du tabac des années 1970 à 1990, qu'on n'a pas par région.

## Conclusion

- À âge égal, les écarts entre départements restent nets (de 130 à 237 patients pour 100 000) : ils ne viennent pas que de la pyramide des âges.
- Au niveau régional, là où le tabac est mesuré, les régions qui fument plus ont plus de cancers du poumon (r entre 0,55 et 0,85 selon l'année). C'est cohérent avec le rôle connu du tabac.
- Mais le tabac régional n'explique rien des écarts **à l'intérieur** des régions, qui sont souvent plus grands que les écarts entre régions.

## Limites

- **Prévalence et non mortalité** : on compte les patients suivis, ce qui dépend aussi de la survie et de l'accès aux soins. Un département qui soigne mieux peut avoir plus de patients.
- **Tabac régional** : 12 points seulement, et aucune donnée pour la Corse, pourtant la plus touchée.
- **Tabac trop récent** : la mesure la plus ancienne date de 2000, alors qu'il faudrait les années 1970-1990. Et « fumeur quotidien » ne dit rien de la quantité fumée ni des anciens fumeurs.
- **Corrélation n'est pas causalité** : on ne contrôle ni les expositions professionnelles (mines, amiante, sidérurgie dans le Nord-Est), ni la pollution de l'air, ni le niveau social. Ces facteurs sont probablement derrière une partie des écarts à l'intérieur des régions.
- **Raisonnement sur des moyennes** : des corrélations entre territoires ne disent rien des individus.

Pour aller plus loin, il faudrait des données de tabac par département et plus anciennes, la mortalité par cancer du poumon (CépiDc) pour s'affranchir de l'effet des soins, et des variables d'exposition professionnelle et de pollution.